# Week 7 Hands-On Lab — Learning to Act, and Learning to Cheat

**ESP3201 · formative hands-on lab.** Runs on free-tier Google Colab with a **CPU runtime**. Trains in seconds.

- **Part A** — tabular **Q-learning** on Gymnasium FrozenLake.
- **Part B** — a **reward-hacking** diagnosis: a mis-specified reward makes the optimal policy farm a proxy instead of solving the task.
- **Part C** — dissect the hacked policy.
- **Part D** — **design and test your own reward fix.**

> **Report only what your run actually produced.** Learning curves are stochastic; quote your run's numbers.

**Attribution.** The Q-learning workflow mirrors the HuggingFace Deep RL Course (Unit 2) and Farama Gymnasium. The reward-hacking framing follows DeepMind's specification-gaming catalogue and OpenAI's *Faulty Reward Functions in the Wild*. All lab code is original to ESP3201.

## Setup

In [ ]:
import sys, os

COURSE_REPO_URL = "https://github.com/bingquan/esp3201-public.git"

try:
    import numpy
except ModuleNotFoundError:
    os.system("pip install -q numpy")
try:
    import gymnasium
except ModuleNotFoundError:
    os.system("pip install -q gymnasium")
try:
    import rl_lab
except ModuleNotFoundError:
    cand = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
    if os.path.exists(os.path.join(cand, "rl_lab.py")):
        sys.path.insert(0, cand)
    elif COURSE_REPO_URL:
        os.system(f"git clone -q {COURSE_REPO_URL} course_repo")
        sys.path.insert(0, "course_repo/mini_assignments/week07_reward_hacking_diagnostics/starter")
    else:
        raise ModuleNotFoundError("rl_lab.py not found. On Colab set COURSE_REPO_URL.")
    import rl_lab
from rl_lab import (q_learning, evaluate, greedy_policy, make_frozenlake,
                    RewardHackGrid, reached_goal_success)
import numpy as np
print("rl_lab loaded.")

## Part A — Q-learning on FrozenLake

In [ ]:
env = make_frozenlake(slippery=False)
ns, na = env.observation_space.n, env.action_space.n
res = q_learning(env, ns, na, episodes=3000, alpha=0.1, gamma=0.95,
                 epsilon=1.0, epsilon_decay=0.999, max_steps=100, seed=0)
print(f"final 100-episode success rate: {res.moving_success_rate(100):.2f}")
ev = evaluate(make_frozenlake(slippery=False), res.Q)
print(f"greedy policy: return={ev['return']:.1f}  success={ev['success']}  steps={len(ev['trajectory'])-1}")

In [ ]:
import matplotlib.pyplot as plt
succ = res.successes(); window = 100
smooth = [sum(succ[max(0,i-window):i+1]) / len(succ[max(0,i-window):i+1]) for i in range(len(succ))]
plt.figure(figsize=(7,3)); plt.plot(smooth)
plt.xlabel('episode'); plt.ylabel(f'success rate ({window}-ep moving)')
plt.title('FrozenLake Q-learning'); plt.tight_layout(); plt.show()

**Try it:** sweep `alpha`, `gamma`, `epsilon_decay`. Which most changes how fast (and whether) the agent learns?

## Part B0 — Q-learning fundamentals: watching a Q-table learn

Part A trained a Q-table and showed you a success-rate curve at the end. This section opens that
up: how do individual Q-values actually move as the agent gathers experience, and how do the
learning-rate and exploration-schedule choices change *how fast* (and whether) that happens?

Every update follows the same rule:

```
Q(s, a)  <-  Q(s, a) + alpha * ( reward + gamma * max_a' Q(s', a') - Q(s, a) )
                                \_____________ TD target _____________/
                       \_________________ TD error __________________/
```

`alpha` (learning rate) controls how much of the TD error gets absorbed into `Q(s, a)` on
each single update. `gamma` (discount) controls how much of the *next* state's best value counts
toward this one. Everything below just watches this one line run, many times, at different settings.

### B0.1 — Watch the value function propagate from the goal

Re-train on FrozenLake, but this time snapshot the whole Q-table every few hundred episodes.
Plotting `V(s) = max_a Q(s, a)` as a 4x4 heatmap at each snapshot lets you watch the *goal's* value
propagate backward through the grid as training progresses -- this is the mechanism behind that
success-rate curve in Part A, not a separate phenomenon.

In [ ]:
env = make_frozenlake(slippery=False)
ns, na = env.observation_space.n, env.action_space.n
snap_res = q_learning(env, ns, na, episodes=3000, alpha=0.1, gamma=0.95,
                      epsilon=1.0, epsilon_decay=0.999, max_steps=100, seed=0,
                      snapshot_every=300)

n_snaps = len(snap_res.snapshots)
fig, axes = plt.subplots(2, (n_snaps + 1) // 2, figsize=(2.2 * ((n_snaps + 1) // 2), 4.4))
for ax, (ep, Q) in zip(axes.flat, snap_res.snapshots):
    V = Q.max(axis=1).reshape(4, 4)
    im = ax.imshow(V, vmin=0, vmax=1, cmap="viridis")
    succ_so_far = snap_res.successes()[:ep + 1]
    window = succ_so_far[-100:]
    ax.set_title(f"ep {ep}\nsucc={sum(window)/len(window):.2f}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes.flat[n_snaps:]:
    ax.axis("off")
fig.suptitle("V(s) = max_a Q(s,a) across training -- watch the goal's value spread backward")
fig.colorbar(im, ax=axes, shrink=0.6, label="V(s)")
plt.show()

**Look for:** which cells light up first, and whether that matches the cells right next to the goal. Cells that never light up are usually holes or cells the greedy policy never needs to pass through.

### B0.2 — Watch individual trajectories update the Q-table

Two single episodes, traced step by step: one from partway through training (still noisy,
still updating a lot), one from near the end (mostly converged, small TD errors). Same update
rule, very different behavior.

In [ ]:
env = make_frozenlake(slippery=False)
traced_res = q_learning(env, ns, na, episodes=3000, alpha=0.1, gamma=0.95,
                        epsilon=1.0, epsilon_decay=0.999, max_steps=100, seed=0,
                        trace_episodes={650, 2500})

def show_trace(ep_idx, label):
    ep_log = traced_res.logs[ep_idx]
    print(f"--- episode {ep_idx} ({label}) -- success={ep_log.success}, {len(ep_log.trace)} steps ---")
    for st in ep_log.trace[:10]:
        print(f"  step {st.step:2d}: state {st.state:2d} -> action {st.action} -> reward {st.reward:+.2f} | "
              f"Q before {st.q_before:6.3f}  TD target {st.td_target:6.3f}  "
              f"TD error {st.td_error:+.3f}  ->  Q after {st.q_after:6.3f}")
    print()

show_trace(650, "mid-training")
show_trace(2500, "near-converged")

### Checkpoint D -- what does a converged update look like?

Compare the TD errors in the two traces. What happens to the TD error as a state-action pair gets visited many times under a stable policy? Does a near-zero TD error mean the Q-value is *correct*, or only that it stopped changing?

### B0.3 — Zoomed out: how do alpha and the exploration schedule affect convergence?

Instead of one run, train a small grid of `(epsilon_decay, alpha)` combinations and record how many
episodes each one needs to reach an 80% moving success rate. This takes under a minute -- tabular
Q-learning is cheap -- and gives a genuinely "zoomed out" view instead of one anecdotal run.

In [ ]:
alphas = [0.02, 0.05, 0.1, 0.3, 0.6]
epsilon_decays = [0.995, 0.998, 0.999, 0.9995, 0.9998]

convergence = np.full((len(epsilon_decays), len(alphas)), np.nan)
for i, ed in enumerate(epsilon_decays):
    for j, a in enumerate(alphas):
        sweep_env = make_frozenlake(slippery=False)
        sweep_res = q_learning(sweep_env, ns, na, episodes=4000, alpha=a, gamma=0.95,
                               epsilon=1.0, epsilon_decay=ed, max_steps=100, seed=0)
        ep_to_converge = sweep_res.episodes_to_reach(0.8, window=100)
        if ep_to_converge is not None:
            convergence[i, j] = ep_to_converge

print("episodes needed to reach an 80% moving success rate (blank = never, within 4000 episodes):")
print("epsilon_decay \\ alpha  " + "  ".join(f"{a:>6}" for a in alphas))
for i, ed in enumerate(epsilon_decays):
    row = "  ".join(f"{convergence[i,j]:6.0f}" if not np.isnan(convergence[i,j]) else "   n/a" for j in range(len(alphas)))
    print(f"{ed:<20}  {row}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
masked = np.ma.masked_invalid(convergence)
cmap = plt.cm.viridis.copy()
cmap.set_bad("lightgray")
im = ax.imshow(masked, cmap=cmap, aspect="auto")
ax.set_xticks(range(len(alphas))); ax.set_xticklabels(alphas)
ax.set_yticks(range(len(epsilon_decays))); ax.set_yticklabels(epsilon_decays)
ax.set_xlabel("alpha (learning rate)"); ax.set_ylabel("epsilon_decay")
ax.set_title("Episodes to reach 80% success (gray = never converged in 4000 episodes)")
fig.colorbar(im, ax=ax, label="episodes to converge")
plt.tight_layout(); plt.show()

### Checkpoint E -- reading the sweep

- Which `epsilon_decay` row never converges, and why -- too much residual randomness, or too little?
- Is there a decay value that is clearly *worse* than both a faster and a slower one? What does that tell you about "more exploration is always safer"?
- Does `alpha` change convergence speed as much as `epsilon_decay` does here? Would you expect that to hold in every environment, or is this specific to FrozenLake's sparse, deterministic reward?

## Part B — A reward you can hack

`RewardHackGrid` is a corridor `[S][ ][P][ ][G]`. Under the **proxy** reward, stepping into the polish tile `P` pays out *every time*. Watch **reward earned** climb while **task success** stays at zero.

In [ ]:
proxy_env = RewardHackGrid(reward_mode='proxy')
hacked = q_learning(proxy_env, proxy_env.n_states, proxy_env.n_actions,
                    episodes=1500, max_steps=proxy_env.max_steps,
                    is_success=reached_goal_success, seed=0)
final_return = sum(hacked.returns()[-100:]) / 100.0
final_success = hacked.moving_success_rate(100)
print(f"MIS-SPECIFIED reward:  mean return={final_return:.1f}   goal-success rate={final_success:.2f}")
ev = evaluate(RewardHackGrid(reward_mode='proxy'), hacked.Q, is_success=reached_goal_success)
print(f"greedy trajectory (cells): {ev['trajectory']}")
print('^ it oscillates around the proxy cell and never reaches cell 4 (G).')

### The built-in fix

`reward_mode='fixed'` removes the proxy payout and adds a small step cost.

In [ ]:
fixed_env = RewardHackGrid(reward_mode='fixed', step_cost=-0.05)
fixed = q_learning(fixed_env, fixed_env.n_states, fixed_env.n_actions,
                   episodes=1500, max_steps=fixed_env.max_steps,
                   is_success=reached_goal_success, seed=0)
ev2 = evaluate(RewardHackGrid(reward_mode='fixed', step_cost=-0.05), fixed.Q, is_success=reached_goal_success)
print(f"FIXED reward:  greedy return={ev2['return']:.2f}  success={ev2['success']}  trajectory={ev2['trajectory']}")

## Part C — Dissect the hacked policy

In [ ]:
print('Greedy action per cell (0=left, 1=right):', greedy_policy(hacked.Q))
for s in range(proxy_env.n_states):
    print(f'  cell {s}: left={hacked.Q[s,0]:7.2f}  right={hacked.Q[s,1]:7.2f}')
print('\nReward spec: entering cell', proxy_env.proxy_cell, 'pays', proxy_env.proxy_reward,
      'every time; goal (cell', proxy_env.goal_cell, ') pays', proxy_env.goal_reward, 'once.')

## Part D — Design and TEST your own reward fix

Don't just propose a fix — implement it and check whether the trained policy still hacks. Write a `custom_reward(prev_pos, new_pos, entered, env)` that returns the reward for one step, then train with `reward_mode='custom'`.

A fix *works* only if the greedy policy **reaches the goal** (`success=True`). Try to make one that does — and try a tempting one that secretly still rewards the proxy and watch it fail.

In [ ]:
def my_reward(prev_pos, new_pos, entered, env):
    # EDIT THIS. Available: env.proxy_cell, env.goal_cell, env.length.
    # Example starting point (a small step cost + a goal bonus, no proxy payout):
    r = -0.05
    if new_pos == env.goal_cell:
        r += 5.0
    return r

env = RewardHackGrid(reward_mode='custom', custom_reward=my_reward)
res = q_learning(env, env.n_states, env.n_actions, episodes=1500,
                 max_steps=env.max_steps, is_success=reached_goal_success, seed=0)
ev = evaluate(RewardHackGrid(reward_mode='custom', custom_reward=my_reward), res.Q,
              is_success=reached_goal_success)
print(f"YOUR reward:  reached_goal={ev['success']}  return={ev['return']:.2f}  trajectory={ev['trajectory']}")
print('A working fix makes reached_goal=True.')

## Worksheet (your deliverable)

### 1. Reward-vs-success table

| Reward design | Mean return | Goal-success rate |
|---------------|-------------|-------------------|
| proxy (mis-specified) | | |
| fixed (built-in) | | |
| **your custom fix** | | |

### 2. Diagnose the hack

- Which cell/action does the hacked greedy policy exploit, and what in the **reward spec** makes it profitable? Cite the Q-values.
- Why is *mean return* a misleading success metric here?

### 3. Your fix

- Paste your `custom_reward`. Did it make `reached_goal=True`?
- If your first attempt failed, what did the policy do instead, and what did you change?
- `What you re-measured to confirm the fix (not just higher reward):`

### 4. Connect to deployment

- In one sentence: where might a real robot reward function hide a proxy like this one?

## AI-Agent Usage Disclosure

State:

- which tools you used
- what they helped produce
- what you verified or rewrote yourself
- one specific thing you did not trust without checking